In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Attention, TimeDistributed, GlobalAveragePooling2D
)
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

# Path to datasets
violence_folder = "/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset/NonViolence"
non_violence_folder = "/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset/Violence"

# Parameters
FRAME_SIZE = (224, 224)  # ResNet input size
FRAMES_PER_VIDEO = 16  # Fixed number of frames per video
BATCH_SIZE = 32
EPOCHS = 20

# Helper function to extract frames from a video
def extract_frames(video_path, num_frames=FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_interval = max(1, total_frames // num_frames)  # To evenly sample frames

    for i in range(num_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * frame_interval)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, FRAME_SIZE)
            frames.append(frame)
    cap.release()
    return np.array(frames)

# Load data and preprocess
def load_data(folder, label):
    video_paths = []
    labels = []
    for video_file in os.listdir(folder):
        video_paths.append(os.path.join(folder, video_file))
        labels.append(label)
    return video_paths, labels

# Combine violence and non-violence videos
violence_videos, violence_labels = load_data(violence_folder, 1)
non_violence_videos, non_violence_labels = load_data(non_violence_folder, 0)

video_paths = violence_videos + non_violence_videos
labels = violence_labels + non_violence_labels

# Split into training and validation sets
video_paths = np.array(video_paths)
labels = np.array(labels)
X_train, X_val, y_train, y_val = train_test_split(video_paths, labels, test_size=0.2, random_state=42, stratify=labels)

In [ ]:
# Load pre-trained ResNet50 model
resnet_weights_path = "/kaggle/input/modd/keras/default/1/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5"
base_model = ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3))
base_model.load_weights(resnet_weights_path)
spatial_model = Model(inputs=base_model.input, outputs=GlobalAveragePooling2D()(base_model.output))

# Preprocess data: Extract frames and compute spatial features
def preprocess_videos(video_paths, labels):
    features = []
    new_labels = []

    for i, video_path in tqdm(enumerate(video_paths), total=len(video_paths)):
        frames = extract_frames(video_path, FRAMES_PER_VIDEO)
        if frames.shape[0] == FRAMES_PER_VIDEO:  # Ensure video has enough frames
            frame_features = spatial_model.predict(frames / 255.0)  # Normalize frames
            features.append(frame_features)
            new_labels.append(labels[i])

    return np.array(features), np.array(new_labels)

X_train_features, y_train = preprocess_videos(X_train, y_train)
X_val_features, y_val = preprocess_videos(X_val, y_val)

# Bi-LSTM with attention mechanism and global pooling
from tensorflow.keras.layers import Lambda

def build_model():
    input_layer = Input(shape=(FRAMES_PER_VIDEO, 2048))  # 2048 = ResNet50 output dimension
    lstm = Bidirectional(LSTM(128, return_sequences=True))(input_layer)
    attention = Attention()([lstm, lstm])  # Self-attention
    pooled = Lambda(lambda x: tf.reduce_mean(x, axis=1))(attention)  # Global Average Pooling
    output = Dense(1, activation="sigmoid")(pooled)  # Binary classification
    model = Model(inputs=input_layer, outputs=output)
    return model

# Build, compile, and train the model
model = build_model()
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
# Train the model
history = model.fit(
    X_train_features, y_train,
    validation_data=(X_val_features, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS
)

# Evaluate the model
loss, accuracy = model.evaluate(X_val_features, y_val)
print(f"Validation Loss: {loss:.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
model.save("/kaggle/working/violence_detection_model.h5")
